# DMOSpeech 2: Preliminary Test of the Proposed Gates
## SNR–Drift Consistency, low-drift veto, guidance contrast and stratified calibration vs SNR-only and SNR + drift

**Thesis:** A Risk-Calibrated Adaptive Handover Method for Teacher-Guided Flow-Matching TTS Inference

The development pilot (EXP-011) found that the 20 dB SNR gate misses 18–22 dB noise and that noise flattens the teacher's trajectory. This notebook tests candidate solutions on development speakers, against the current rules:

| Policy | Rule (candidates Steps 4/5/6, reference Step 8, SNR gate 20 dB) |
|---|---|
| Reference | Step 8 for every input |
| **SNR-only** | Step 4 if prompt SNR ≥ 20 dB, else Step 8 (current gate) |
| **SNR + drift** | Definition 3.4: first candidate with d_k ≤ λ (the proposal as written) |
| Low-drift veto | SNR gate, then fall back if d₄ < λ_low (two-sided stabilization window) |
| SNR–Drift Consistency (SDC) gate | fall back if log d₄ is far below what the prompt's SNR predicts (fitted on clean prompts) |
| Guidance-contrast gate | fall back on the guidance gap g₄ = ‖v_c − v_u‖ / ‖v_c‖ (direction set on training folds) |
| SDC + guidance | fall back if either fires |

Every parameter is chosen on training folds (5-fold speaker cross-validation) by fixed-sequence **Learn-Then-Test**, either **pooled** over conditions or **stratified** (certified only if every condition passes: an intersection–union test). Results are out of fold, per condition.

**Two sizes** (switch `QUICK` in Step 6):

| Mode | Speakers × conditions | New synthesis | Teacher-only pass | T4 time (est.) |
|---|---|---|---|---|
| **Quick** (default) | 60 × 6: clean, noisy15, Gaussian 18/20/22/25 dB | 40 speakers × 4 Gaussian levels = 160 cases | 360 cases | about 75 min |
| Full | 100 × all 11 (adds babble, reverberation, telephone) | 80 speakers × 9 conditions = 720 cases | 1,100 cases | about 3–4 h |

Existing labels are reused (the T4 grid for clean/noisy15; the pilot for its 20 speakers × 11 conditions), so only the missing cases are synthesized. Times are estimates from the pilot's 15.6 s per case. Quick mode covers the failure the pilot found (the 18–22 dB band around the SNR gate); Full adds the non-Gaussian degradations. Each mode writes to its own Drive folder. Development speakers only: no held-out data is used. The settings are written to `preregistration.json` before any synthesis. Results are flushed to Google Drive after every case, so the run resumes after a disconnect.

---
### Instructions
1. Set the runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. **Runtime > Run all.** Step 7 runs in the background; if the browser disconnects, reconnect and run **Step 7b**.
3. The last cell downloads `DMOSpeech2_Consistency_Gate_Results.zip`.

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Repositories & Install Dependencies
`Shonget-DMOSpeech2` is checked out at `8a7868f`, the commit whose `core/` sampler produced the T4 grid (it also contains the `656d86f` fix that makes SIM honour the ground-truth sample rate). The thesis repository (branch `rhenelAudit`) supplies the runner, the split manifests and the T4 grid; if it is private, add a Colab secret named `GITHUB_TOKEN`.

If Colab shows **"Restart session"** after the installs, click it, then continue from **Step 3** (no need to re-run Step 2). Every later step re-enters the repository folder and re-reads its settings, so a restart is safe.

In [ ]:
import os, subprocess
DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}
!git log --oneline -1

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
THESIS_BRANCH = 'rhenelAudit'   # the branch that holds the follow-up runner
if not os.path.exists('/content/Final-Thesis-Setup'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git",
                    '/content/Final-Thesis-Setup'], check=True)
!git -C /content/Final-Thesis-Setup fetch -q origin {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup checkout -q {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup pull -q origin {THESIS_BRANCH}
!git -C /content/Final-Thesis-Setup log --oneline -1

# Same pinned dependencies as the 1,000-utterance GPU retest notebook
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")

runner_path = '/content/Final-Thesis-Setup/code/colab/run_followup_tests.py'
if not os.path.exists(runner_path):
    raise FileNotFoundError(f"{runner_path} is not on branch {THESIS_BRANCH}. Push it to GitHub before continuing.")

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`, checked against the SHA-256 hashes recorded for the earlier runs.

In [ ]:
%cd /content/Shonget-DMOSpeech2
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

import hashlib
for name, digest in {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
                     'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f"{name}: SHA-256 mismatch (delete it and re-run this step)"
    print(f"{name}: SHA-256 verified")

### Step 4: Download the Seed-TTS test-en Dataset
Prompt and ground-truth audio for all 666 speakers: the 500 in the splits and the 166 outside them.

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav
!wc -l data/seed-tts-eval/en/meta.lst

### Step 5: Download DEMAND Cafeteria Babble
Real recorded noise for the `babble15` / `babble20` conditions: DEMAND (Thiemann, Ito and Vincent, 2013), environment PCAFETER, 16 kHz, channel 1, from Zenodo record 1227121. The runner records its SHA-256 with the results.

In [ ]:
import os
DEMAND = '/content/demand'
if not os.path.exists(f'{DEMAND}/PCAFETER/ch01.wav'):
    os.makedirs(DEMAND, exist_ok=True)
    !wget -q -c -O /content/PCAFETER_16k.zip "https://zenodo.org/records/1227121/files/PCAFETER_16k.zip?download=1"
    !unzip -q -o /content/PCAFETER_16k.zip -d {DEMAND}
if not os.path.exists(f'{DEMAND}/PCAFETER/ch01.wav'):
    !find {DEMAND} -maxdepth 3 | head -20
    raise FileNotFoundError("PCAFETER/ch01.wav not found: check the archive layout printed above")
!ls -lh {DEMAND}/PCAFETER/ch01.wav

### Step 6: Choose the Mode + Smoke Test (2 development speakers, a few minutes)
Set `QUICK = True` (about 75 min) or `QUICK = False` (Full, about 3–4 h). The choice is saved to Drive, so later cells and a resumed run use the same mode.

The smoke test synthesizes 2 non-pilot development speakers under Gaussian 20 dB and telephone, and records the teacher signals (including clean) for both. Writes to a **separate** `smoke/` folder.

In [ ]:
QUICK = True   # True: 60 speakers x 6 conditions (~75 min); False: 100 speakers x 11 conditions (~3-4 h)
import os, json
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/dmospeech2_results', exist_ok=True)
json.dump({'quick': QUICK}, open('/content/drive/MyDrive/dmospeech2_results/consistency_run_config.json', 'w'))

# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
cfg_path = os.path.join(persist, 'consistency_run_config.json')
QUICK = json.load(open(cfg_path))['quick'] if os.path.exists(cfg_path) else True   # set in Step 6
OUT = os.path.join(persist, 'consistency_gate_prelim_' + ('quick' if QUICK else 'full'))
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_consistency_gate_prelim.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"{'Quick' if QUICK else 'Full'} mode. Results in Google Drive: {OUT}")

!python {RUNNER} smoke --out "{OUT}" --dmo_root {DMO} --demand_dir {DEMAND}

import pandas as pd
sig = pd.read_csv(f'{OUT}/smoke/signals.csv')
print(f"Smoke test OK: {len(sig)} signal rows")
display(sig[['utt_id', 'condition', 'prompt_snr', 'd_4', 'd_5', 'd_6', 'g_4']])

### Step 7: Full Run + Analysis
Writes `preregistration.json` (once), synthesizes the missing labels, records the teacher signals for every case (360 Quick, 1,100 Full), then runs the analysis.

The job runs **in the background** and writes its full output to `MyDrive/dmospeech2_results/consistency_gate_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running; reconnect and run **Step 7b**.

**If the Colab machine itself is reset**, the results so far are safe in Google Drive (`MyDrive/dmospeech2_results/consistency_gate_prelim_quick/` or `_full/`). Re-run Steps 1–5, skip Step 6, then run this cell again: it continues from the last saved case.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
cfg_path = os.path.join(persist, 'consistency_run_config.json')
QUICK = json.load(open(cfg_path))['quick'] if os.path.exists(cfg_path) else True   # set in Step 6
OUT = os.path.join(persist, 'consistency_gate_prelim_' + ('quick' if QUICK else 'full'))
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_consistency_gate_prelim.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"{'Quick' if QUICK else 'Full'} mode. Results in Google Drive: {OUT}")

import subprocess, time, re
log_path = os.path.join(persist, 'consistency_gate_log.txt')
cmd = f'python {RUNNER} run --out "{OUT}" --dmo_root {DMO} --demand_dir {DEMAND}' + (' --quick' if QUICK else '')
running = subprocess.run(['pgrep', '-f', 'run_consistency_gate_prelim.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f"A job is already running (pid {pid}); monitoring it instead of starting a new one.")
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{log_path}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f"Started (pid {proc.pid}). Full log: {log_path}")

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(log_path, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 7b: Check Progress After a Reconnect
Shows whether the background job is still running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, json, subprocess
persist = '/content/drive/MyDrive/dmospeech2_results'
cfg_path = os.path.join(persist, 'consistency_run_config.json')
QUICK = json.load(open(cfg_path))['quick'] if os.path.exists(cfg_path) else True
OUT = os.path.join(persist, 'consistency_gate_prelim_' + ('quick' if QUICK else 'full'))
print('Mode:', 'Quick' if QUICK else 'Full')
log_path = os.path.join(persist, 'consistency_gate_log.txt')
running = subprocess.run(['pgrep', '-f', 'run_consistency_gate_prelim.py'], capture_output=True, text=True).stdout.split()
print("Job running:", "yes (pid " + ", ".join(running) + ")" if running else "no")
if os.path.exists(log_path):
    lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print("Latest progress:", prog[-1] if prog else "(none yet)")
    for line in lines[-5:]:
        print(line)
for name, per_case, total in (('labels_new.csv', 4, 160 if QUICK else 720), ('signals.csv', 1, 360 if QUICK else 1100)):
    path = os.path.join(OUT, name)
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
    print(f"{name}: about {n // per_case} of {total} cases saved")

### Step 8: View the Report
If the run stopped early, add `--allow_partial` for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
cfg_path = os.path.join(persist, 'consistency_run_config.json')
QUICK = json.load(open(cfg_path))['quick'] if os.path.exists(cfg_path) else True   # set in Step 6
OUT = os.path.join(persist, 'consistency_gate_prelim_' + ('quick' if QUICK else 'full'))
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_consistency_gate_prelim.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"{'Quick' if QUICK else 'Full'} mode. Results in Google Drive: {OUT}")

from IPython.display import Markdown, display
rep = os.path.join(OUT, 'CONSISTENCY_GATE_REPORT.md')
if not os.path.exists(rep) and os.path.exists(os.path.join(OUT, 'signals.csv')):
    !python {RUNNER} analyze --out "{OUT}"
if os.path.exists(rep):
    display(Markdown(open(rep, encoding='utf-8').read()))
else:
    print(f"No report yet at {rep}. Run Step 7 first.")

### Step 9: Zip All Outputs and Download
Packs the label and signal CSVs, preregistration, results JSON, report and the runner into one ZIP, also saved to Drive. Failure audio in `listening/` stays on Drive.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
cfg_path = os.path.join(persist, 'consistency_run_config.json')
QUICK = json.load(open(cfg_path))['quick'] if os.path.exists(cfg_path) else True   # set in Step 6
OUT = os.path.join(persist, 'consistency_gate_prelim_' + ('quick' if QUICK else 'full'))
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_consistency_gate_prelim.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"{'Quick' if QUICK else 'Full'} mode. Results in Google Drive: {OUT}")

import zipfile, subprocess, datetime
zip_name = 'DMOSpeech2_Consistency_Gate_Results.zip'
with open(os.path.join(OUT, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"dmospeech2_commit: {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"thesis_commit: {subprocess.run(['git', '-C', THESIS, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(OUT):
        dirs[:] = [d for d in dirs if d != 'listening']
        for fname in files_:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
!cp "{zip_name}" "{persist}/"
print(f"Also saved to Google Drive: {persist}/{zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")